In [1]:
import os

# --- Fix: make protobuf runtime deterministic BEFORE any TF/pydicom imports.
# We avoid offline pip installs (often unavailable) and instead force pure-Python protobuf,
# then purge any already-imported protobuf modules to prevent the GetPrototype crash.
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
os.environ.setdefault("PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION_VERSION", "2")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import sys
import warnings

warnings.filterwarnings("ignore")


def _purge_protobuf_modules():
    for k in list(sys.modules.keys()):
        if (
            k == "google"
            or k.startswith("google.")
            or k.startswith("google.protobuf")
            or k.startswith("google._upb")
            or k.startswith("protobuf")
        ):
            del sys.modules[k]


_purge_protobuf_modules()

import numpy as np
import pandas as pd

try:
    import cv2

    _HAS_CV2 = True
except Exception:
    _HAS_CV2 = False
    from skimage.transform import resize as _sk_resize

_HAS_TF = True
tf = None
keras = None
dicom = None


def _try_import_pydicom_then_tf():
    """Import pydicom first, then TensorFlow. This ordering is most robust for this Kaggle image."""
    global tf, keras, dicom, _HAS_TF

    # pydicom first
    try:
        import pydicom as _dicom

        dicom = _dicom
    except Exception as e:
        dicom = None
        tf = None
        keras = None
        _HAS_TF = False
        return False, f"pydicom import failed: {repr(e)}"

    # then TF/Keras
    try:
        import tensorflow as _tf
        from tensorflow import keras as _keras

        tf, keras = _tf, _keras
        _HAS_TF = True
        return True, None
    except Exception as e:
        tf = None
        keras = None
        _HAS_TF = False
        return False, f"TensorFlow import failed: {repr(e)}"


ok, err = _try_import_pydicom_then_tf()
if not ok:
    # One clean retry after purging protobuf modules again (handles rare pre-imports from environment)
    _purge_protobuf_modules()
    ok2, err2 = _try_import_pydicom_then_tf()
    if not ok2:
        raise RuntimeError(
            "Failed to import pydicom/TensorFlow after protobuf purge/retry. "
            f"First error={err} Second error={err2}"
        )

SEED = 42
np.random.seed(SEED)
if _HAS_TF:
    tf.random.set_seed(SEED)
    try:
        tf.config.threading.set_intra_op_parallelism_threads(1)
        tf.config.threading.set_inter_op_parallelism_threads(1)
    except Exception:
        pass

print("TensorFlow available:", _HAS_TF)
if _HAS_TF:
    print("TensorFlow:", tf.__version__)
print("Has cv2:", _HAS_CV2)
print("pydicom available:", dicom is not None)



2026-05-13 15:38:45.909989: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778686725.920989      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778686725.924280      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

TensorFlow available: True
TensorFlow: 2.18.0
Has cv2: True
pydicom available: True


In [2]:
BASE_CANDIDATES = [
    "../input/rsna-miccai-brain-tumor-radiogenomic-classification",
    "/kaggle/input/rsna-miccai-brain-tumor-radiogenomic-classification",
    "/kaggle/data/rsna-miccai-brain-tumor-radiogenomic-classification",
    "/kaggle/input/rsna-miccai-brain-tumor-radiogenomic-classification/rsna-miccai-brain-tumor-radiogenomic-classification",
    "/kaggle/data/rsna-miccai-brain-tumor-radiogenomic-classification/rsna-miccai-brain-tumor-radiogenomic-classification",
]

BASE = None
for c in BASE_CANDIDATES:
    if os.path.isdir(c):
        BASE = c
        break

if BASE is None:
    raise FileNotFoundError(
        "Could not find competition dataset folder in expected locations."
    )

TRAIN_DIR = os.path.join(BASE, "train")
TEST_DIR = os.path.join(BASE, "test")
LABELS_CSV = os.path.join(BASE, "train_labels.csv")
SAMPLE_SUB = os.path.join(BASE, "sample_submission.csv")

assert os.path.isdir(TEST_DIR), f"Missing test dir: {TEST_DIR}"
assert os.path.isfile(SAMPLE_SUB), f"Missing sample submission: {SAMPLE_SUB}"
assert os.path.isfile(LABELS_CSV), f"Missing labels csv: {LABELS_CSV}"

sample_sub = pd.read_csv(SAMPLE_SUB)
train_labels = pd.read_csv(LABELS_CSV)

sample_sub["BraTS21ID"] = sample_sub["BraTS21ID"].astype(str).str.zfill(5)
train_labels["BraTS21ID"] = train_labels["BraTS21ID"].astype(str).str.zfill(5)

BAD_IDS = set(["00109", "00123", "00709"])
train_labels = train_labels[~train_labels["BraTS21ID"].isin(BAD_IDS)].reset_index(
    drop=True
)

print("BASE:", BASE)
print("Train labels:", train_labels.shape, " Sample submission:", sample_sub.shape)



BASE: ../input/rsna-miccai-brain-tumor-radiogenomic-classification
Train labels: (523, 2)  Sample submission: (59, 2)


In [3]:
from functools import lru_cache


def _percentile_linear_1d(x: np.ndarray, q: float) -> float:
    x = x.reshape(-1)
    n = x.size
    if n == 0:
        return 0.0
    if n == 1:
        return float(x[0])
    pos = (q / 100.0) * (n - 1)
    lo = int(np.floor(pos))
    hi = int(np.ceil(pos))
    if lo == hi:
        return float(np.partition(x, lo)[lo])
    xc = x.copy()
    v_lo = float(np.partition(xc, lo)[lo])
    v_hi = float(np.partition(xc, hi)[hi])
    w = pos - lo
    return v_lo * (1.0 - w) + v_hi * w


def _safe_dcm_pixel_array(dcm_path: str):
    """Read DICOM and return pixel array as float32, or None on failure."""
    try:
        ds = dicom.dcmread(
            dcm_path,
            force=True,
            stop_before_pixels=False,
            specific_tags=[
                "PixelData",
                "Rows",
                "Columns",
                "BitsAllocated",
                "BitsStored",
                "HighBit",
                "PixelRepresentation",
                "SamplesPerPixel",
                "PhotometricInterpretation",
                "RescaleIntercept",
                "RescaleSlope",
                "TransferSyntaxUID",
            ],
        )
        arr = ds.pixel_array.astype(np.float32)
        if arr.size == 0:
            return None
        return arr
    except Exception:
        return None


@lru_cache(maxsize=4096)
def _scan_modality_folder(case_dir: str, modality: str):
    """Return modality folder path for a case, matching by folder name (robust to sorting/index issues)."""
    p = os.path.join(case_dir, modality)
    if os.path.isdir(p):
        return p
    mod_l = modality.lower()
    try:
        with os.scandir(case_dir) as it:
            for e in it:
                if e.is_dir() and e.name.lower() == mod_l:
                    return e.path
    except FileNotFoundError:
        return None
    return None


def _list_dcm_files_fast(mod_dir: str, take_k: int):
    """Deterministically get up to take_k DICOM file paths in the same order as sorted(...)[:take_k], faster."""
    names = []
    try:
        with os.scandir(mod_dir) as it:
            for e in it:
                if e.is_file() and e.name.lower().endswith(".dcm"):
                    names.append(e.name)
    except FileNotFoundError:
        return []

    if not names:
        return []

    names.sort()
    if len(names) > take_k:
        names = names[:take_k]
    return [os.path.join(mod_dir, n) for n in names]


def _resize_2d_float(arr: np.ndarray, out_hw):
    out_h, out_w = out_hw
    if _HAS_CV2:
        return cv2.resize(arr, (out_w, out_h), interpolation=cv2.INTER_LINEAR).astype(
            np.float32, copy=False
        )
    return _sk_resize(
        arr, (out_h, out_w), preserve_range=True, anti_aliasing=True
    ).astype(np.float32)


def extract_case_features(
    case_dir: str,
    modalities=("FLAIR", "T1w", "T1wCE", "T2w"),
    img_px_size=128,
    max_slices=12,
):
    """
    Minimal feature extraction from DICOM: for each modality, pick up to `max_slices` informative slices,
    compute summary stats. Designed to run within time limits.
    """
    # --- Fix: fail fast with clear message if pydicom didn't import (prevents cryptic later crashes).
    if dicom is None:
        raise RuntimeError("pydicom is not available; cannot extract DICOM features.")

    feats = []
    for mod in modalities:
        mod_dir = _scan_modality_folder(case_dir, mod)
        if mod_dir is None:
            feats.extend([0.0, 0.0, 0.0, 0.0, 0.0, 0.0])
            continue

        dcm_files = _list_dcm_files_fast(mod_dir, take_k=max_slices * 8)

        if not dcm_files:
            feats.extend([0.0, 0.0, 0.0, 0.0, 0.0, 0.0])
            continue

        slice_stats = []
        for fp in dcm_files:
            arr = _safe_dcm_pixel_array(fp)
            if arr is None:
                continue

            s = float(arr.sum())
            if s <= 100000:
                continue

            arr_rs = _resize_2d_float(arr, (img_px_size, img_px_size))
            mx = float(np.max(arr_rs))
            if mx <= 0:
                continue
            arr_n = arr_rs / mx
            if float(arr_n.sum()) <= 1000:
                continue

            p10 = _percentile_linear_1d(arr_n, 10.0)
            p90 = _percentile_linear_1d(arr_n, 90.0)

            slice_stats.append(
                (
                    float(arr_n.mean()),
                    float(arr_n.std()),
                    float(arr_n.min()),
                    float(arr_n.max()),
                    float(p10),
                    float(p90),
                    float(arr_n.sum()),
                )
            )
            if len(slice_stats) >= max_slices:
                break

        if not slice_stats:
            feats.extend([0.0, 0.0, 0.0, 0.0, 0.0, 0.0])
        else:
            slice_stats = np.asarray(slice_stats, dtype=np.float32)
            feats.extend(
                [
                    float(slice_stats[:, 0].mean()),  # mean of means
                    float(slice_stats[:, 1].mean()),  # mean of stds
                    float(slice_stats[:, 4].mean()),  # mean p10
                    float(slice_stats[:, 5].mean()),  # mean p90
                    float(slice_stats[:, 6].mean()),  # mean sum
                    float(len(slice_stats)),  # count of slices used
                ]
            )
    return np.asarray(feats, dtype=np.float32)


def _feature_worker(args):
    i, sid, root_dir = args
    case_dir = os.path.join(root_dir, sid)
    return i, extract_case_features(case_dir)


def build_feature_table(root_dir: str, ids: list):
    from concurrent.futures import ProcessPoolExecutor, as_completed

    n = len(ids)
    X = np.zeros((n, 4 * 6), dtype=np.float32)

    max_workers = min(8, (os.cpu_count() or 2))

    try:
        with ProcessPoolExecutor(max_workers=max_workers) as ex:
            futures = [
                ex.submit(_feature_worker, (i, sid, root_dir))
                for i, sid in enumerate(ids)
            ]
            done = 0
            for fut in as_completed(futures):
                i, feats = fut.result()
                X[i] = feats
                done += 1
                if done % 20 == 0:
                    print(f"Processed {done}/{n} cases")
        return X
    except Exception as e:
        print("ProcessPool failed, falling back to single-process. Error:", repr(e))
        for i, sid in enumerate(ids):
            _, feats = _feature_worker((i, sid, root_dir))
            X[i] = feats
            if (i + 1) % 20 == 0:
                print(f"Processed {i+1}/{n} cases")
        return X




In [4]:
MODEL_DIR_CANDIDATES = [
    "../input/trained-model-for-rsnamiccai",
    "/kaggle/input/trained-model-for-rsnamiccai",
]
MODEL_DIR = None
for c in MODEL_DIR_CANDIDATES:
    if os.path.isdir(c):
        MODEL_DIR = c
        break

pretrained_paths = []
if MODEL_DIR is not None:
    pretrained_paths = [
        os.path.join(MODEL_DIR, "model_rsna_miccai_100epochs.h5"),
        os.path.join(MODEL_DIR, "rsna_miccai_100_epochs_T1W.h5"),
        os.path.join(MODEL_DIR, "rsna_miccai_100_epochs_T1wCE.h5"),
        os.path.join(MODEL_DIR, "rsna_miccai_100_epochs_T2W.h5"),
        os.path.join(MODEL_DIR, "rsna_miccai_200_epochs_flair.h5"),
    ]
pretrained_paths = [p for p in pretrained_paths if os.path.isfile(p)]

pretrained_models = []
if _HAS_TF and pretrained_paths:
    for p in pretrained_paths:
        try:
            pretrained_models.append(keras.models.load_model(p, compile=False))
        except Exception as e:
            print(f"Failed loading {p}: {e}")

print("Found pretrained models:", len(pretrained_models))



Found pretrained models: 0


In [5]:
train_ids = train_labels["BraTS21ID"].tolist()
y = train_labels["MGMT_value"].astype(np.float32).values

X_train = build_feature_table(TRAIN_DIR, train_ids)

mu = X_train.mean(axis=0, keepdims=True)
sigma = X_train.std(axis=0, keepdims=True) + 1e-6
X_train_z = (X_train - mu) / sigma

baseline_model = None
if _HAS_TF:
    if len(pretrained_models) == 0:
        baseline_model = keras.Sequential(
            [
                keras.layers.Input(shape=(X_train_z.shape[1],)),
                keras.layers.Dense(1, activation="sigmoid"),
            ]
        )
        baseline_model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=1e-2),
            loss="binary_crossentropy",
        )
        baseline_model.fit(X_train_z, y, epochs=30, batch_size=32, verbose=0)
        print("Trained baseline model on features.")
    else:
        print("Using pretrained model(s); skipping baseline training.")
else:
    print("TensorFlow unavailable; skipping model training/loading.")



Processed 20/523 cases
Processed 40/523 cases
Processed 60/523 cases
Processed 80/523 cases
Processed 100/523 cases
Processed 120/523 cases
Processed 140/523 cases
Processed 160/523 cases
Processed 180/523 cases
Processed 200/523 cases
Processed 220/523 cases
Processed 240/523 cases
Processed 260/523 cases
Processed 280/523 cases
Processed 300/523 cases
Processed 320/523 cases
Processed 340/523 cases
Processed 360/523 cases
Processed 380/523 cases
Processed 400/523 cases
Processed 420/523 cases
Processed 440/523 cases
Processed 460/523 cases
Processed 480/523 cases
Processed 500/523 cases
Processed 520/523 cases


2026-05-13 15:39:23.374263: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Trained baseline model on features.


In [6]:
test_ids = sample_sub["BraTS21ID"].tolist()
X_test = build_feature_table(TEST_DIR, test_ids)
X_test_z = (X_test - mu) / sigma

all_preds = []

if _HAS_TF and len(pretrained_models) > 0:
    for m in pretrained_models:
        p = m.predict(X_test_z, verbose=0).reshape(-1).astype(np.float32)
        all_preds.append(p)

if _HAS_TF and baseline_model is not None:
    p = baseline_model.predict(X_test_z, verbose=0).reshape(-1).astype(np.float32)
    all_preds.append(p)

if not all_preds:
    test_pred = np.full((len(test_ids),), 0.5, dtype=np.float32)
else:
    test_pred = np.mean(np.stack(all_preds, axis=0), axis=0).astype(np.float32)

test_pred = np.clip(test_pred, 1e-6, 1 - 1e-6)

print(
    "Pred stats:",
    float(test_pred.min()),
    float(test_pred.mean()),
    float(test_pred.max()),
)



Processed 20/59 cases
Processed 40/59 cases
Pred stats: 0.33274760842323303 0.549902081489563 0.6976315975189209


In [7]:
sub_df = pd.DataFrame(
    {
        "BraTS21ID": sample_sub["BraTS21ID"].astype(str).str.zfill(5),
        "MGMT_value": test_pred.astype(float),
    }
)

assert list(sub_df.columns) == ["BraTS21ID", "MGMT_value"]
assert len(sub_df) == len(sample_sub)

print(sub_df.head())
print(sub_df.describe())

sub_path = "submission.csv"
sub_df.to_csv(sub_path, index=False)
print("Wrote:", sub_path, "rows:", len(sub_df))

  BraTS21ID  MGMT_value
0     00356    0.469908
1     00140    0.591635
2     00757    0.597060
3     00275    0.405732
4     00570    0.556928
       MGMT_value
count   59.000000
mean     0.549902
std      0.083384
min      0.332748
25%      0.496683
50%      0.582329
75%      0.600344
max      0.697632
Wrote: submission.csv rows: 59
